In [44]:
from kfp import dsl
import kfp
from kfp.dsl import Dataset, Input, Output
from kfp import compiler
from kfp import kubernetes

In [45]:

@dsl.component(base_image="python:3.11", packages_to_install=["loguru"])
def get_paths_data(data_dir: str, paths_dataset: Output[Dataset], path_labels: Output[Dataset]):
    
    import os
    import json
    from loguru import logger

    logger.debug(f"Data directory: {data_dir}")

    label_names = []
    for d in os.listdir(data_dir):
        if os.path.isdir(os.path.join(data_dir, d)):
            label_names.append(d)

    with open(path_labels.path, "w") as f:
        for label in label_names:
            f.write(label + "\n")
    
    with open(paths_dataset.path, "w") as f:
        for label in label_names:
            label_dir = os.path.join(data_dir, label)
            for fname in os.listdir(label_dir):
                full_path = os.path.join(label_dir, fname)
                logger.debug(f"Processing file: {full_path} with label: {label}")
                record = {
                    "path": full_path,
                    "label": label
                }
                f.write(json.dumps(record) + "\n")

    logger.debug(f"Saved paths dataset to {paths_dataset.path}")


In [46]:
@dsl.component(base_image="python:3.11", packages_to_install=["opencv-python-headless"])
def compute_image_stats(paths_dataset: Input[Dataset], path_stats_dataset: Output[Dataset]):
    
    import os
    import json
    import cv2

    def compute_stats(path, label) -> dict:
        img = cv2.imread(path)
        if img is None:
            return None
        stats = {}
        for i, color in enumerate(['Blue', 'Green', 'Red']):
            channel = img[:, :, i]
            stats[f'{color}_mean'] = float(channel.mean())
            stats[f'{color}_std'] = float(channel.std())
            stats[f'{color}_min'] = float(channel.min())
            stats[f'{color}_max'] = float(channel.max())
        img_num = os.path.basename(path).split("_")[1].split('.')[0]
        return {
            'image_id': img_num,
            'label': label,
            **stats
        }

    # Read paths and labels from previous component
    path_label_pairs = []
    with open(paths_dataset.path, "r") as f:
        for line in f:
            record = json.loads(line.strip())
            path_label_pairs.append((record["path"], record["label"]))

    # Compute stats for each image
    results = []
    for path, label in path_label_pairs:
        result = compute_stats(path, label)
        if result is not None:
            results.append(result)

    # Filter out None (bad images)
    results = [r for r in results if r is not None]

    # Save as JSONL
    with open(path_stats_dataset.path, "w") as f:
        for record in results:
            f.write(json.dumps(record) + "\n")

    print(f"Processed {len(results)} images out of {len(path_label_pairs)}")

In [47]:
@dsl.component(base_image="python:3.11", packages_to_install=["pandas", "scikit-learn", "loguru"])
def split_train_test(path_stats_dataset: Input[Dataset], path_labels: Input[Dataset], 
                     train_dataset: Output[Dataset], test_dataset: Output[Dataset]):

    import pandas as pd
    from sklearn.model_selection import train_test_split
    from loguru import logger
    import json

    # Load data stats
    records = []
    with open(path_stats_dataset.path, "r") as f:
        for line in f:
            record = json.loads(line.strip())
            records.append(record)

    # Load label names    label_names = []
    with open(path_labels.path, "r") as f:
        label_names = [line.strip() for line in f]

    # Create DataFrame
    df = pd.DataFrame(records)
    df.set_index('image_id', inplace=True)

    map_labels2idx = {label: idx for idx, label in enumerate(label_names)}
    logger.debug(f"Label to index mapping: {map_labels2idx}")
    map_idx2labels = {idx: label for label, idx in map_labels2idx.items()}
    logger.debug(f"Index to label mapping: {map_idx2labels}")

    # Map labels to indices
    df["label_idx"] = df["label"].map(map_labels2idx)

    # Prepare data for stratified split
    X = df[df.columns.difference(['label','label_idx'])].values
    y = df['label_idx'].values

    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # Save train and test datasets
    train_df = pd.DataFrame(X_train, columns=df.columns.difference(['label','label_idx']))
    train_df['label_idx'] = y_train
    train_df['label'] = train_df['label_idx'].map(map_idx2labels)
    train_df.to_csv(train_dataset.path, index=False)

    test_df = pd.DataFrame(X_test, columns=df.columns.difference(['label','label_idx']))
    test_df['label_idx'] = y_test
    test_df['label'] = test_df['label_idx'].map(map_idx2labels)
    test_df.to_csv(test_dataset.path, index=False)  


In [48]:
@dsl.component(base_image="python:3.11", packages_to_install=["pandas", "scikit-learn", "loguru"])
def hyperparams_serach(train_dataset: Input[Dataset], best_params: Output[Dataset]):

    import pandas as pd
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.model_selection import RandomizedSearchCV
    import json

    # Load datasets training set
    train_df = pd.read_csv(train_dataset.path)

    # Prepare training data
    X_train = train_df[train_df.columns.difference(['label','label_idx'])].values
    y_train = train_df['label_idx'].values

    param_dist = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1],
    'max_depth': [2, 3]
    }

    hyp_param_search = RandomizedSearchCV(
        estimator=GradientBoostingClassifier(random_state=42),
        param_distributions=param_dist,
        n_iter=5,
        cv=3,  # stratified 5-fold cross-validation
        random_state=42,
        n_jobs=-1
    )
    hyp_param_search.fit(X_train, y_train)
    print("Best Hyperparameters:", hyp_param_search.best_params_)
    print("Best CV Score:", hyp_param_search.best_score_)

    # append seed to best params
    hyp_param_search.best_params_['random_state'] = 42

    with open(best_params.path, "w") as f:
        json.dump(hyp_param_search.best_params_, f)


In [49]:
# # Train final model with best hyperparameters
# best_model = GradientBoostingClassifier(**hyp_param_search.best_params_, random_state=42)
# best_model.fit(X_train, y_train)

@dsl.component(base_image="python:3.11", packages_to_install=["pandas", "scikit-learn", "loguru"])
def train_final_model(train_dataset: Input[Dataset], best_params: Input[Dataset], model_output: Output[Dataset]):

    import pandas as pd
    from sklearn.ensemble import GradientBoostingClassifier
    import json
    from loguru import logger

    # Load datasets training set
    train_df = pd.read_csv(train_dataset.path)

    # Prepare training data
    X_train = train_df[train_df.columns.difference(['label','label_idx'])].values
    y_train = train_df['label_idx'].values

    with open(best_params.path, "r") as f:
        best_hyperparams = json.load(f)

    best_model = GradientBoostingClassifier(**best_hyperparams)
    best_model.fit(X_train, y_train)

    # Save the model (you can use joblib or pickle)
    import joblib
    joblib.dump(best_model, model_output.path)
    logger.debug(f"Saved trained model to {model_output.path}")



In [50]:
@dsl.pipeline(name="ETL Pipeline")
def ETL_pipeline(data_dir: str):
    
    # Step 1: Get paths and labels
    get_paths_task = get_paths_data(data_dir=data_dir)
    # Mount the PVC into the component pod
    get_paths_task.set_caching_options(False)
    kubernetes.mount_pvc(
        get_paths_task,
        pvc_name="kind-local-pvc",
        mount_path="/data/persistent",
    )

    # Step 2: Compute image statistics
    stats_task = compute_image_stats(paths_dataset=get_paths_task.outputs["paths_dataset"])
    # Mount the PVC into the component pod
    stats_task.set_caching_options(False)
    kubernetes.mount_pvc(
        stats_task,
        pvc_name="kind-local-pvc",
        mount_path="/data/persistent",
    )

    # Step 3: Split into train and test sets
    split_task = split_train_test(
        path_stats_dataset=stats_task.outputs["path_stats_dataset"],
        path_labels=get_paths_task.outputs["path_labels"]
    )
    # Mount the PVC into the component pod
    split_task.set_caching_options(False)
    kubernetes.mount_pvc(
        split_task,
        pvc_name="kind-local-pvc",
        mount_path="/data/persistent",
    )

    # Step 4: Hyperparameter search
    train_task = hyperparams_serach(train_dataset=split_task.outputs["train_dataset"])
    # Mount the PVC into the component pod
    train_task.set_caching_options(False)
    kubernetes.mount_pvc(
        train_task,
        pvc_name="kind-local-pvc",
        mount_path="/data/persistent",
    )

    # Step 5: Train final model with best hyperparameters
    final_model_task = train_final_model(
        train_dataset=split_task.outputs["train_dataset"],
        best_params=train_task.outputs["best_params"]
    )
    # Mount the PVC into the component pod
    final_model_task.set_caching_options(False)
    kubernetes.mount_pvc(
        final_model_task,
        pvc_name="kind-local-pvc",
        mount_path="/data/persistent",
    )


In [51]:
# Compile the pipeline
compiler.Compiler().compile(
    pipeline_func=ETL_pipeline,
    package_path="etl_pipeline.yaml",
)

# Create a KFP client and run the pipeline
client = kfp.Client("http://localhost:8888")
run = client.create_run_from_pipeline_package(
    pipeline_file='etl_pipeline.yaml',
    arguments={'data_dir': '/data/persistent/datasets/EuroSAT_RGB'},
    run_name='etl-run',
    experiment_name='etl-experiment',
)
print(run.run_id)

/opt/homebrew/Caskroom/miniconda/base/envs/remote/lib/python3.10/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


2dd1159e-6244-4e12-af31-99e3b37713e9


In [52]:
# from minio import Minio

# client = Minio("localhost:9000", 
#                access_key="minio", 
#                secret_key="minio123", 
#                secure=False)
# response = client.fget_object(
#     bucket_name="mlpipeline",
#     object_name="path/to/your/object",
#     file_path="./stats_dataset.jsonl"
# )
# print(response)